# 3. Anime and user vector visualization

This notebook diagnoses the content representations used by the shared-protocol recommender.

The plots inspect genre and franchise structure, differences between user profiles, and score separation between positives and distractors.

Two-dimensional projections can distort distance. Check them against cosine similarities and neighbours in the original vector space.

## Visualized representations

| Configuration | Vector |
| --- | --- |
| C1 | Genre TF-IDF |
| C3 | Genre TF-IDF + type one-hot + episode bucket |
| C4 | Genre TF-IDF + title TF-IDF |

The notebook uses all 7,223 anime and a reproducible sample of 5,000 users.

Projection strategy:

```text
Sparse vectors
    ↓
Truncated SVD
    ↓
UMAP with cosine distance, when umap-learn is available
    ↓
2D coordinates
```

If UMAP is unavailable, deterministic two-dimensional Truncated SVD is used and explicitly reported.

In [ ]:
from __future__ import annotations

import html
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

SEED = 42
USER_SAMPLE_SIZE = 5_000
EVALUATION_USER_SAMPLE_SIZE = 5_000
QUALITY_SAMPLE_SIZE = 1_000
RANDOM_PAIR_COUNT = 20_000

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "data.py").exists():
            return path
    raise FileNotFoundError("Run this notebook inside the dam repository.")

REPO_ROOT = find_repo_root()
os.environ.setdefault("MPLCONFIGDIR", "/tmp/dam-matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, normalize
from sklearn.decomposition import TruncatedSVD

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    build_observed_sets,
    iterative_positive_k_core,
    leave_two_out_split,
    load_anime_ratings,
    positive_interactions,
    remove_ambiguous_pairs,
    sample_evaluation_candidates,
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.2,
    "font.size": 10,
})

print("Repository:", REPO_ROOT)

## 1. Recreate the shared training population

This repeats the exact data boundary used by the recommender:

- ambiguous pairs removed
- ratings 7–10 treated as positive
- iterative positive 5-core
- seed-42 leave-two-out split

In [ ]:
def resolve_dataset_dir() -> Path:
    configured = os.getenv("ANIME_DATASET_DIR")
    if configured:
        path = Path(configured).expanduser().resolve()
    else:
        try:
            import kagglehub
        except ImportError as exc:
            raise ImportError(
                "Install kagglehub or set ANIME_DATASET_DIR."
            ) from exc
        path = Path(
            kagglehub.dataset_download("CooperUnion/anime-recommendations-database")
        )
    for filename in ("anime.csv", "rating.csv"):
        if not (path / filename).exists():
            raise FileNotFoundError(path / filename)
    return path

DATASET_DIR = resolve_dataset_dir()
anime_raw = pd.read_csv(DATASET_DIR / "anime.csv")
ratings_raw = load_anime_ratings(DATASET_DIR / "rating.csv")

deduplicated, _ = remove_ambiguous_pairs(ratings_raw)
positive_rows = positive_interactions(deduplicated, 7)
positive_core, _ = iterative_positive_k_core(positive_rows, 5)
split = leave_two_out_split(positive_core, SEED)

eligible_users = sorted(int(value) for value in split.train["user_id"].unique())
warm_items = sorted(int(value) for value in split.train["anime_id"].unique())

assert len(eligible_users) == 60_384
assert len(warm_items) == 7_223
assert len(split.train) == 5_087_394

metadata = (
    anime_raw.drop_duplicates("anime_id")
    .set_index("anime_id")
    .reindex(warm_items)
    .reset_index()
)
assert metadata["name"].notna().all()

print(f"Training positives: {len(split.train):,}")
print(f"Users: {len(eligible_users):,}")
print(f"Anime: {len(warm_items):,}")

## 2. Rebuild C1, C3, and C4 anime vectors

The same encoders and normalization rules as the recommendation notebook are used.

In [ ]:
def split_genres(text: str) -> list[str]:
    if not text or pd.isna(text):
        return []
    return [value.strip() for value in str(text).split(",") if value.strip()]

def clean_title(text: str) -> str:
    return re.sub(r"\s+", " ", html.unescape(str(text)).lower()).strip()

EPISODE_BUCKETS = ["1", "2-6", "7-13", "14-26", "27-52", "53+", "Unknown"]

def episode_bucket(value: object) -> str:
    number = pd.to_numeric(value, errors="coerce")
    if pd.isna(number):
        return "Unknown"
    number = int(number)
    if number == 1:
        return "1"
    if number <= 6:
        return "2-6"
    if number <= 13:
        return "7-13"
    if number <= 26:
        return "14-26"
    if number <= 52:
        return "27-52"
    return "53+"

genre_vectorizer = TfidfVectorizer(
    analyzer=split_genres,
    lowercase=False,
    norm=None,
    dtype=np.float32,
)
genre_block = genre_vectorizer.fit_transform(metadata["genre"].fillna(""))

type_encoder = OneHotEncoder(
    categories=[["Movie", "Music", "ONA", "OVA", "Special", "TV"]],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)
type_block = type_encoder.fit_transform(metadata[["type"]].fillna("Unknown"))

episode_encoder = OneHotEncoder(
    categories=[EPISODE_BUCKETS],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)
episode_block = episode_encoder.fit_transform(
    metadata["episodes"].map(episode_bucket).to_frame("episode_bucket")
)

title_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    lowercase=False,
    norm=None,
    dtype=np.float32,
)
title_block = title_vectorizer.fit_transform(metadata["name"].map(clean_title))

raw_item_vectors = {
    "C1": genre_block,
    "C3": sparse.hstack(
        [genre_block, type_block, episode_block], format="csr"
    ),
    "C4": sparse.hstack(
        [genre_block, title_block], format="csr"
    ),
}
item_vectors = {
    name: normalize(matrix, axis=1).tocsr().astype(np.float32)
    for name, matrix in raw_item_vectors.items()
}

dimensions = pd.DataFrame({
    "configuration": list(item_vectors),
    "dimensions": [matrix.shape[1] for matrix in item_vectors.values()],
    "nonzero values": [matrix.nnz for matrix in item_vectors.values()],
})
display(dimensions)

assert item_vectors["C1"].shape == (7_223, 43)
assert item_vectors["C3"].shape == (7_223, 56)
assert item_vectors["C4"].shape == (7_223, 6_721)

## 3. Build rating-weighted user vectors

Each user vector is the normalized weighted sum of positive training-anime vectors.

```text
rating 7 → weight 1
rating 8 → weight 2
rating 9 → weight 3
rating 10 → weight 4
```

In [ ]:
user_to_index = {
    user_id: index for index, user_id in enumerate(eligible_users)
}
item_to_index = {
    item_id: index for index, item_id in enumerate(warm_items)
}

user_item_weights = sparse.csr_matrix(
    (
        split.train["rating"].to_numpy(dtype=np.float32) - 6,
        (
            split.train["user_id"].map(user_to_index).to_numpy(),
            split.train["anime_id"].map(item_to_index).to_numpy(),
        ),
    ),
    shape=(len(eligible_users), len(warm_items)),
    dtype=np.float32,
)

def build_profiles(features: sparse.csr_matrix) -> sparse.csr_matrix:
    return normalize(
        user_item_weights @ features,
        axis=1,
    ).tocsr().astype(np.float32)

user_vectors = {
    name: build_profiles(features)
    for name, features in item_vectors.items()
}

rng = np.random.default_rng(SEED)
sampled_user_indices = np.sort(
    rng.choice(
        len(eligible_users),
        size=min(USER_SAMPLE_SIZE, len(eligible_users)),
        replace=False,
    )
)
sampled_user_ids = np.asarray(eligible_users)[sampled_user_indices]
user_activity = np.asarray(user_item_weights.getnnz(axis=1)).ravel()

print(f"Visualizing {len(sampled_user_indices):,} sampled users.")

## 4. Projection helper and quality checks

Two checks accompany each map:

- **Trustworthiness:** whether nearby 2D points were also neighbours originally
- **Neighbour overlap:** fraction of original top-10 neighbours preserved in 2D

Higher values mean the projection preserves more of the original neighbourhood structure.

In [ ]:
try:
    import umap

    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

def project_to_2d(
    matrix: sparse.csr_matrix,
    seed: int = SEED,
) -> tuple[np.ndarray, str]:
    max_components = min(50, matrix.shape[0] - 1, matrix.shape[1] - 1)
    if UMAP_AVAILABLE:
        reduced = TruncatedSVD(
            n_components=max_components,
            random_state=seed,
        ).fit_transform(matrix)
        coordinates = umap.UMAP(
            n_components=2,
            n_neighbors=30,
            min_dist=0.15,
            metric="cosine",
            random_state=seed,
        ).fit_transform(reduced)
        return coordinates, f"SVD-{max_components} + UMAP"

    coordinates = TruncatedSVD(
        n_components=2,
        random_state=seed,
    ).fit_transform(matrix)
    return coordinates, "Truncated SVD"

def neighbour_overlap(
    original: sparse.csr_matrix,
    projected: np.ndarray,
    neighbours: int = 10,
) -> float:
    original_indices = NearestNeighbors(
        n_neighbors=neighbours + 1,
        metric="cosine",
        algorithm="brute",
    ).fit(original).kneighbors(original, return_distance=False)[:, 1:]
    projected_indices = NearestNeighbors(
        n_neighbors=neighbours + 1,
        metric="euclidean",
    ).fit(projected).kneighbors(projected, return_distance=False)[:, 1:]
    overlaps = [
        len(set(left).intersection(right)) / neighbours
        for left, right in zip(original_indices, projected_indices)
    ]
    return float(np.mean(overlaps))

def projection_quality(
    matrix: sparse.csr_matrix,
    coordinates: np.ndarray,
    sample_size: int = QUALITY_SAMPLE_SIZE,
) -> dict[str, float]:
    size = min(sample_size, matrix.shape[0])
    indices = np.random.default_rng(SEED).choice(
        matrix.shape[0], size=size, replace=False
    )
    original_sample = matrix[indices]
    projected_sample = coordinates[indices]
    dense_sample = original_sample.toarray()
    return {
        "trustworthiness@10": float(
            trustworthiness(
                dense_sample,
                projected_sample,
                n_neighbors=10,
                metric="cosine",
            )
        ),
        "top10_neighbour_overlap": neighbour_overlap(
            original_sample,
            projected_sample,
            neighbours=10,
        ),
    }

print("UMAP available:", UMAP_AVAILABLE)

## 5. Anime-vector maps

Colour represents the first listed genre. Only the most frequent genre groups receive separate colours so the plot remains readable.

In [ ]:
def primary_genre(value: object) -> str:
    genres = split_genres(value)
    return genres[0] if genres else "Missing"

def compact_groups(values: pd.Series, maximum: int = 8) -> pd.Series:
    keep = set(values.value_counts().head(maximum).index)
    return values.where(values.isin(keep), "Other")

anime_genre_group = compact_groups(metadata["genre"].map(primary_genre))
genre_groups = sorted(anime_genre_group.unique())
colour_map = {
    group: plt.cm.tab10(index % 10)
    for index, group in enumerate(genre_groups)
}

anime_coordinates = {}
anime_projection_rows = []

for name, matrix in item_vectors.items():
    coordinates, method = project_to_2d(matrix)
    anime_coordinates[name] = coordinates
    anime_projection_rows.append({
        "configuration": name,
        "method": method,
        **projection_quality(matrix, coordinates),
    })

display(pd.DataFrame(anime_projection_rows).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for axis, name in zip(axes, ["C1", "C3", "C4"]):
    coordinates = anime_coordinates[name]
    for group in genre_groups:
        mask = anime_genre_group.eq(group).to_numpy()
        axis.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=8,
            alpha=0.55,
            color=colour_map[group],
            label=group,
            linewidths=0,
        )
    axis.set_title(f"{name} anime vectors")
    axis.set_xlabel("Projection dimension 1")
    axis.set_ylabel("Projection dimension 2")

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, frameon=False)
fig.suptitle("Anime-vector spaces: genre, metadata, and title effects")
fig.tight_layout(rect=(0, 0.10, 1, 0.94))
plt.show()

### How to read the maps

- Broad C1 regions indicate genre structure.
- C3 changes show whether type and episode length separate anime further.
- Tight C4 islands may represent franchises rather than broad semantic content.
- Do not compare absolute axis values across panels. Each projection is fitted separately.

## 6. User-vector maps

Users are coloured by the largest genre coordinate in their C1 profile. Marker size represents the number of positive training anime.

In [ ]:
genre_names = np.asarray(genre_vectorizer.get_feature_names_out())
sampled_c1 = user_vectors["C1"][sampled_user_indices]
dominant_genre_index = np.asarray(sampled_c1.argmax(axis=1)).ravel()
dominant_genre = pd.Series(genre_names[dominant_genre_index])
user_genre_group = compact_groups(dominant_genre)
user_groups = sorted(user_genre_group.unique())
user_colour_map = {
    group: plt.cm.tab10(index % 10)
    for index, group in enumerate(user_groups)
}
sample_sizes = 8 + 18 * (
    np.log1p(user_activity[sampled_user_indices])
    / np.log1p(user_activity[sampled_user_indices]).max()
)

user_coordinates = {}
user_projection_rows = []

for name, profiles in user_vectors.items():
    sampled_profiles = profiles[sampled_user_indices]
    coordinates, method = project_to_2d(sampled_profiles)
    user_coordinates[name] = coordinates
    user_projection_rows.append({
        "configuration": name,
        "method": method,
        **projection_quality(sampled_profiles, coordinates),
    })

display(pd.DataFrame(user_projection_rows).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for axis, name in zip(axes, ["C1", "C3", "C4"]):
    coordinates = user_coordinates[name]
    for group in user_groups:
        mask = user_genre_group.eq(group).to_numpy()
        axis.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=sample_sizes[mask],
            alpha=0.45,
            color=user_colour_map[group],
            label=group,
            linewidths=0,
        )
    axis.set_title(f"{name} user profiles")
    axis.set_xlabel("Projection dimension 1")
    axis.set_ylabel("Projection dimension 2")

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, frameon=False)
fig.suptitle("User-profile spaces; marker size represents positive-history size")
fig.tight_layout(rect=(0, 0.10, 1, 0.94))
plt.show()

## 7. Original-space user similarity

This diagnostic does not use the 2D coordinates.

It measures cosine similarity between random user pairs in the original feature spaces. A distribution concentrated near 1 indicates weak personalization because many user profiles point in similar directions.

In [ ]:
pair_rng = np.random.default_rng(SEED)
left = pair_rng.integers(0, len(eligible_users), size=RANDOM_PAIR_COUNT)
right = pair_rng.integers(0, len(eligible_users), size=RANDOM_PAIR_COUNT)

pairwise_user_similarity = {}
summary_rows = []

for name, profiles in user_vectors.items():
    similarities = np.asarray(
        profiles[left].multiply(profiles[right]).sum(axis=1)
    ).ravel()
    pairwise_user_similarity[name] = similarities
    summary_rows.append({
        "configuration": name,
        "mean": similarities.mean(),
        "median": np.median(similarities),
        "p90": np.quantile(similarities, 0.90),
    })

display(pd.DataFrame(summary_rows).round(4))

bins = np.linspace(0, 1, 51)
for name, similarities in pairwise_user_similarity.items():
    plt.hist(
        similarities,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        label=name,
    )
plt.xlabel("Cosine similarity between random user profiles")
plt.ylabel("Density")
plt.title("Are user profiles too similar?")
plt.legend(frameon=False)
plt.show()

## 8. Hidden-positive versus distractor scores

For reproducibility without materializing all 6,038,400 validation candidates, the notebook uses the first 5,000 sorted users.

Because candidate generation also processes users in sorted order, these are the exact same validation candidates those users receive in the full protocol.

In [ ]:
diagnostic_users = eligible_users[:min(
    EVALUATION_USER_SAMPLE_SIZE,
    len(eligible_users),
)]
diagnostic_validation = split.validation.loc[
    split.validation["user_id"].isin(diagnostic_users)
]
observed_by_user = build_observed_sets(
    deduplicated,
    diagnostic_users,
)
diagnostic_candidates = sample_evaluation_candidates(
    held_out=diagnostic_validation,
    observed_by_user=observed_by_user,
    warm_items=warm_items,
    negative_count=99,
    seed=SEED,
)

assert len(diagnostic_candidates) == len(diagnostic_users) * 100
assert diagnostic_candidates.groupby("user_id")["label"].sum().eq(1).all()

candidate_user_index = diagnostic_candidates["user_id"].map(
    user_to_index
).to_numpy(dtype=np.int32)
candidate_item_index = diagnostic_candidates["item_id"].map(
    item_to_index
).to_numpy(dtype=np.int32)

def candidate_scores(
    profiles: sparse.csr_matrix,
    features: sparse.csr_matrix,
) -> np.ndarray:
    return np.asarray(
        profiles[candidate_user_index]
        .multiply(features[candidate_item_index])
        .sum(axis=1)
    ).ravel()

score_distributions = {}
score_summary_rows = []

for name in ["C1", "C3", "C4"]:
    scores = candidate_scores(user_vectors[name], item_vectors[name])
    positive = scores[diagnostic_candidates["label"].eq(1).to_numpy()]
    distractor = scores[diagnostic_candidates["label"].eq(0).to_numpy()]
    score_distributions[name] = (positive, distractor)

    score_frame = diagnostic_candidates[["user_id", "label"]].copy()
    score_frame["score"] = scores
    positive_by_user = (
        score_frame.loc[score_frame["label"].eq(1)]
        .set_index("user_id")["score"]
    )
    distractor_rows = score_frame.loc[score_frame["label"].eq(0)].copy()
    distractor_rows["positive_score"] = distractor_rows["user_id"].map(
        positive_by_user
    )
    pairwise_win_rate = float(
        (distractor_rows["positive_score"] > distractor_rows["score"]).mean()
    )

    score_summary_rows.append({
        "configuration": name,
        "positive median": np.median(positive),
        "distractor median": np.median(distractor),
        "positive>distractor rate": pairwise_win_rate,
    })

display(pd.DataFrame(score_summary_rows).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for axis, name in zip(axes, ["C1", "C3", "C4"]):
    positive, distractor = score_distributions[name]
    combined = np.concatenate([positive, distractor])
    low, high = np.quantile(combined, [0.005, 0.995])
    bins = np.linspace(low, high, 55)
    axis.hist(
        distractor,
        bins=bins,
        density=True,
        alpha=0.55,
        label="Unobserved distractor",
    )
    axis.hist(
        positive,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2.2,
        label="Hidden positive",
    )
    axis.set_title(name)
    axis.set_xlabel("Cosine score")
    axis.set_ylabel("Density")

axes[-1].legend(frameon=False)
fig.suptitle("Can content similarity separate positives from distractors?")
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## 9. Nearest-anime examples

Nearest neighbours are calculated in the original vector spaces, not from the 2D plots. This reveals what each configuration considers similar.

In [ ]:
requested_anchors = [
    "Steins;Gate",
    "Death Note",
    "Naruto",
    "Kimi no Na wa.",
]
available_anchors = [
    name for name in requested_anchors
    if metadata["name"].eq(name).any()
]

neighbour_rows = []

for configuration, features in item_vectors.items():
    model = NearestNeighbors(
        n_neighbors=6,
        metric="cosine",
        algorithm="brute",
    ).fit(features)

    for anchor_name in available_anchors:
        anchor_row = int(metadata.index[metadata["name"].eq(anchor_name)][0])
        distances, indices = model.kneighbors(features[anchor_row])
        for rank, (distance, item_index) in enumerate(
            zip(distances[0][1:], indices[0][1:]),
            start=1,
        ):
            neighbour_rows.append({
                "configuration": configuration,
                "anchor": anchor_name,
                "rank": rank,
                "neighbour": metadata.iloc[item_index]["name"],
                "cosine similarity": 1 - float(distance),
                "genre": metadata.iloc[item_index]["genre"],
                "type": metadata.iloc[item_index]["type"],
            })

neighbour_table = pd.DataFrame(neighbour_rows)
display(neighbour_table.round({"cosine similarity": 4}))

## 10. Joint user–anime view

The C4 item vectors and sampled user vectors are projected together so their positions are comparable.

One median-activity user is highlighted together with:

- five highest-rated training anime
- the validation positive
- the three highest-scoring validation candidates

In [ ]:
# Restrict the joint-view users to the exact candidate-diagnostic prefix.
joint_user_indices = sampled_user_indices[
    sampled_user_indices < len(diagnostic_users)
][:1_000]
joint_user_count = len(joint_user_indices)
assert joint_user_count > 0
joint_profiles = user_vectors["C4"][joint_user_indices]
joint_matrix = sparse.vstack(
    [item_vectors["C4"], joint_profiles],
    format="csr",
)
joint_coordinates, joint_method = project_to_2d(joint_matrix)

item_joint = joint_coordinates[:len(warm_items)]
user_joint = joint_coordinates[len(warm_items):]

joint_activity = user_activity[joint_user_indices]
example_local_index = int(
    np.argsort(joint_activity)[len(joint_activity) // 2]
)
example_user_index = int(joint_user_indices[example_local_index])
example_user_id = int(eligible_users[example_user_index])

example_history = (
    split.train.loc[split.train["user_id"].eq(example_user_id)]
    .sort_values(["rating", "anime_id"], ascending=[False, True])
    .head(5)
)
history_item_indices = [
    item_to_index[int(item_id)]
    for item_id in example_history["anime_id"]
]

example_heldout_item = int(
    split.validation.loc[
        split.validation["user_id"].eq(example_user_id),
        "anime_id",
    ].iloc[0]
)
heldout_index = item_to_index[example_heldout_item]

example_candidates = diagnostic_candidates.loc[
    diagnostic_candidates["user_id"].eq(example_user_id)
].copy()
example_candidate_indices = example_candidates["item_id"].map(
    item_to_index
).to_numpy(dtype=np.int32)
example_scores = np.asarray(
    user_vectors["C4"][example_user_index]
    .multiply(item_vectors["C4"][example_candidate_indices])
    .sum(axis=1)
).ravel()
example_candidates["score"] = example_scores
top_candidate_indices = [
    item_to_index[int(item_id)]
    for item_id in example_candidates.nlargest(3, "score")["item_id"]
]

print("Projection:", joint_method)
print("Example user:", example_user_id)
display(
    example_history.merge(
        metadata[["anime_id", "name"]],
        on="anime_id",
        how="left",
    )[["anime_id", "name", "rating"]]
)
print("Validation positive:", metadata.iloc[heldout_index]["name"])
display(
    example_candidates.nlargest(3, "score").merge(
        metadata[["anime_id", "name"]].rename(columns={"anime_id": "item_id"}),
        on="item_id",
        how="left",
    )[["item_id", "name", "label", "score"]]
)

In [ ]:
fig, axis = plt.subplots(figsize=(11, 7))

axis.scatter(
    item_joint[:, 0],
    item_joint[:, 1],
    s=7,
    alpha=0.18,
    color="gray",
    label="Anime",
)
axis.scatter(
    user_joint[:, 0],
    user_joint[:, 1],
    s=12,
    alpha=0.30,
    color="tab:blue",
    label="Sampled users",
)
axis.scatter(
    user_joint[example_local_index, 0],
    user_joint[example_local_index, 1],
    s=130,
    marker="*",
    color="tab:red",
    label=f"User {example_user_id}",
    zorder=5,
)
axis.scatter(
    item_joint[history_item_indices, 0],
    item_joint[history_item_indices, 1],
    s=60,
    color="tab:orange",
    label="Top-rated training anime",
    zorder=4,
)
axis.scatter(
    item_joint[heldout_index, 0],
    item_joint[heldout_index, 1],
    s=100,
    marker="D",
    color="tab:green",
    label="Validation positive",
    zorder=5,
)
axis.scatter(
    item_joint[top_candidate_indices, 0],
    item_joint[top_candidate_indices, 1],
    s=75,
    marker="s",
    facecolors="none",
    edgecolors="tab:purple",
    linewidths=1.8,
    label="Top-3 candidates",
    zorder=5,
)

user_point = user_joint[example_local_index]
for item_index in set(
    history_item_indices + [heldout_index] + top_candidate_indices
):
    axis.plot(
        [user_point[0], item_joint[item_index, 0]],
        [user_point[1], item_joint[item_index, 1]],
        color="gray",
        alpha=0.25,
        linewidth=0.8,
    )

axis.set_xlabel("Projection dimension 1")
axis.set_ylabel("Projection dimension 2")
axis.set_title("Joint C4 user–anime projection")
axis.legend(frameon=False, loc="best")
plt.show()

## 11. Feature-block magnitude diagnostic

Measure each block's share of squared vector magnitude before final normalization. Concatenated blocks can have different magnitudes even without explicit weights.

In [ ]:
def squared_norm(matrix: sparse.csr_matrix) -> np.ndarray:
    return np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel()

genre_energy = squared_norm(genre_block)
type_energy = squared_norm(type_block)
episode_energy = squared_norm(episode_block)
title_energy = squared_norm(title_block)

c3_total = genre_energy + type_energy + episode_energy
c4_total = genre_energy + title_energy

block_share_rows = []
for configuration, blocks, total in [
    (
        "C3",
        {
            "genre": genre_energy,
            "type": type_energy,
            "episodes": episode_energy,
        },
        c3_total,
    ),
    (
        "C4",
        {
            "genre": genre_energy,
            "title": title_energy,
        },
        c4_total,
    ),
]:
    for block, energy in blocks.items():
        share = np.divide(
            energy,
            total,
            out=np.zeros_like(energy),
            where=total > 0,
        )
        block_share_rows.append({
            "configuration": configuration,
            "block": block,
            "mean share": share.mean(),
            "median share": np.median(share),
        })

block_shares = pd.DataFrame(block_share_rows)
display(block_shares.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, configuration in zip(axes, ["C3", "C4"]):
    current = block_shares.loc[
        block_shares["configuration"].eq(configuration)
    ]
    axis.bar(current["block"], current["median share"])
    axis.set_ylim(0, 1)
    axis.set_ylabel("Median squared-magnitude share")
    axis.set_title(configuration)
fig.suptitle("Implicit feature-block weighting before final normalization")
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

## Reading the diagnostics

- **Anime maps:** Are clusters understandable from genre, type, or franchise?
- **User maps:** Are there distinct preference regions or one dense cloud?
- **User similarity:** Are random users already highly similar?
- **Score distributions:** Do hidden positives separate from distractors?
- **Neighbours:** Does C4 retrieve semantically related anime or only franchise names?
- **Block shares:** Are added features numerically strong enough to affect cosine similarity?
- **Projection quality:** Are the two-dimensional neighbourhoods trustworthy?

The original cosine similarities, score distributions, and nearest-neighbour tables are more reliable than visual distance alone.